# Molecule CV — Stats Comparison

Compares the three CNN architectures on pIC50 regression error.

**Inputs** (from `train_and_save.ipynb` on Colab, dropped into `molecule_cv_results/`):
- `predictions.csv` — `CID`, `pIC50_actual`, `Class`, `pred_admet`, `pred_custom_cnn`, `pred_toxic_colors`
- `model_metrics.csv` — headline metrics per model

**Analyses:**
1. Wilcoxon signed-rank on per-molecule absolute errors, pairwise + Holm correction
2. Residual analysis — scatter, distributions, per-model error correlation
3. Stacked metalearner — linear combination of the three predictions, 5-fold CV

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RESULTS_DIR = 'molecule_cv_results'

## Load Predictions

In [ ]:
preds = pd.read_csv(os.path.join(RESULTS_DIR, 'predictions.csv'))
metrics = pd.read_csv(os.path.join(RESULTS_DIR, 'model_metrics.csv'))

MODEL_COLS = [c for c in preds.columns if c.startswith('pred_')]
MODEL_NAMES = {c: c.replace('pred_', '').replace('_', ' ').title() for c in MODEL_COLS}

print(f"Test molecules: {len(preds)}")
print(f"Models: {list(MODEL_NAMES.values())}\n")
print("Headline metrics from Colab run:")
print(metrics.to_string(index=False))

## 1. Wilcoxon Signed-Rank on Absolute Errors

For each pair of models, tests whether per-molecule absolute errors on pIC50 differ systematically. Holm-Bonferroni correction across the three pairwise tests.

In [ ]:
y_true = preds['pIC50_actual'].values

abs_err = pd.DataFrame({
    MODEL_NAMES[c]: np.abs(y_true - preds[c].values) for c in MODEL_COLS
})
print("Per-model MAE (sanity check — should match model_metrics.csv):")
print(abs_err.mean().round(4).to_string())

In [ ]:
# Pairwise Wilcoxon signed-rank with Holm-Bonferroni correction
pairs = [(a, b) for i, a in enumerate(MODEL_COLS) for b in MODEL_COLS[i+1:]]

wil_rows = []
for a, b in pairs:
    err_a = np.abs(y_true - preds[a].values)
    err_b = np.abs(y_true - preds[b].values)
    stat, p = wilcoxon(err_a, err_b)
    wil_rows.append({
        'model_a': MODEL_NAMES[a],
        'model_b': MODEL_NAMES[b],
        'mae_a': round(err_a.mean(), 4),
        'mae_b': round(err_b.mean(), 4),
        'median_abs_err_diff': round(np.median(err_a - err_b), 4),
        'w_stat': round(stat, 2),
        'p_value': p,
    })

# Holm-Bonferroni correction
k = len(wil_rows)
order = sorted(range(k), key=lambda i: wil_rows[i]['p_value'])
running_max = 0.0
p_holm = [None] * k
for rank, idx in enumerate(order):
    adj = (k - rank) * wil_rows[idx]['p_value']
    running_max = max(running_max, adj)
    p_holm[idx] = min(running_max, 1.0)

for row, p_adj in zip(wil_rows, p_holm):
    row['p_holm'] = round(p_adj, 6)
    row['p_value'] = round(row['p_value'], 6)
    row['sig'] = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'ns'

wil_df = pd.DataFrame(wil_rows)
wil_df

## 2. Residual Analysis

In [ ]:
# Actual vs predicted scatter — one subplot per model
fig, axes = plt.subplots(1, len(MODEL_COLS), figsize=(5 * len(MODEL_COLS), 5), sharey=True)
lo, hi = y_true.min() - 0.5, y_true.max() + 0.5

for ax, col in zip(axes, MODEL_COLS):
    y_pred = preds[col].values
    ax.scatter(y_true, y_pred, alpha=0.5, s=15)
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3, label='y = x')
    mae = np.mean(np.abs(y_true - y_pred))
    r2 = r2_score(y_true, y_pred)
    ax.set_title(f"{MODEL_NAMES[col]}\nMAE={mae:.3f}, R²={r2:.3f}")
    ax.set_xlabel('Actual pIC50')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.legend(loc='upper left', fontsize=8)
axes[0].set_ylabel('Predicted pIC50')
fig.tight_layout()
plt.show()

In [ ]:
# Signed-residual distributions
residuals = pd.DataFrame({
    MODEL_NAMES[c]: y_true - preds[c].values for c in MODEL_COLS
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name in residuals.columns:
    axes[0].hist(residuals[name], bins=30, alpha=0.4, label=name)
axes[0].axvline(0, color='k', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residual Distributions')
axes[0].legend()

axes[1].boxplot([residuals[n] for n in residuals.columns], labels=list(residuals.columns))
axes[1].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Residual (actual - predicted)')
axes[1].set_title('Residual Spread')
fig.tight_layout()
plt.show()

print("\nResidual summary:")
print(residuals.describe().round(3))

In [ ]:
# Error correlation between models
# High correlation => models fail on the same molecules (less complementary)
# Low correlation => independent failure modes (more potential for stacking to help)
print("Absolute error correlation between models:")
print(abs_err.corr().round(3))

## 3. Stacked Metalearner

Linear combination of the three base predictions. Two things worth looking at:

1. **Coefficients** from a full-test-set fit — diagnostic. Are they all mixed (real complementary signal) or concentrated on one model (that model dominates)?
2. **5-fold CV MAE** — honest out-of-sample estimate of stacked performance. Compare to the best single model.

In [ ]:
X_meta = preds[MODEL_COLS].values
y_meta = y_true

# Full-fit coefficients (diagnostic)
lr_full = LinearRegression().fit(X_meta, y_meta)
coef_df = pd.DataFrame({
    'model': [MODEL_NAMES[c] for c in MODEL_COLS],
    'coef': lr_full.coef_.round(4),
})
print(f"Intercept: {lr_full.intercept_:.4f}")
print(coef_df.to_string(index=False))

In [ ]:
# 5-fold CV evaluation of stacked predictions
kf = KFold(n_splits=5, shuffle=True, random_state=42)
stacked_preds = np.zeros(len(y_meta))
for train_idx, test_idx in kf.split(X_meta):
    lr = LinearRegression().fit(X_meta[train_idx], y_meta[train_idx])
    stacked_preds[test_idx] = lr.predict(X_meta[test_idx])

# Comparison table — base models vs stacked
comparison_rows = []
for col in MODEL_COLS:
    y_pred = preds[col].values
    comparison_rows.append({
        'model': MODEL_NAMES[col],
        'mae': round(mean_absolute_error(y_meta, y_pred), 4),
        'mse': round(mean_squared_error(y_meta, y_pred), 4),
        'r2': round(r2_score(y_meta, y_pred), 4),
    })
comparison_rows.append({
    'model': 'Stacked (5-fold CV)',
    'mae': round(mean_absolute_error(y_meta, stacked_preds), 4),
    'mse': round(mean_squared_error(y_meta, stacked_preds), 4),
    'r2': round(r2_score(y_meta, stacked_preds), 4),
})
pd.DataFrame(comparison_rows)